# CMT — large300 — Nouveaux objectifs de stage

Notebook construit à partir des 5 axes notés en réunion :

1. **Visualisation** — afficher à un temps donné (+ évolution temporelle) : flux de Reynolds, cisaillement, vent $u$, vent $w$, $\partial_z u$, $\partial_z^2 u$.
2. **Equation discovery (STLSQ/SINDy)** avec des termes physiques *grande échelle* uniquement (pas de termes de turbulence petite échelle type SGS).
3. **Robustesse train/test dans le temps** — le split actuel (train/test sur différents pas de temps) est probablement biaisé car les grandeurs (flux de Reynolds) dérivent au cours du temps → CV temporelle + pistes ML alternatives.
4. **Fonction de courant $\psi$ moyennée en $y$ → décomposition POD/ACP** : combien de modes garder ? reconstruction de $u,w$ à partir des modes, contenu énergétique.
5. **Circulation** — cellules fermées de $\psi$ (signe unique, part de 0 et revient à 0) → décomposition en cellules de circulation exactes.

Ce notebook reprend le **setup commun** déjà utilisé dans les notebooks précédents (`6_CMT_trois_objectifs`, etc.) : lecture NetCDF niveau par niveau (RAM), $\rho_0$ via température virtuelle / gaz parfait, masques PRW humide/sec, catégories up/dn/env, solveur de Poisson périodique-x / Dirichlet-z pour $\psi$.


## 0. Setup commun

In [ ]:

import numpy as np
import xarray as xr
import netCDF4 as nc
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.optimize import lstsq as sp_lstsq
from scipy.signal import find_peaks
from sklearn.decomposition import PCA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from skimage.measure import label, regionprops
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10


In [ ]:

# ---- Config à adapter -------------------------------------------------
DATA_DIR   = Path("/data/large300")          # dossier des fichiers Méso-NH
FILE_PATTERN = "large300.1.*.nc"              # pattern des fichiers instantanés
OUT_DIR    = Path("./outputs_new_objectifs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

G      = 9.81          # m/s^2
RD     = 287.06         # J/kg/K, gaz parfait air sec
RV     = 461.5          # J/kg/K, gaz parfait vapeur d'eau
CP     = 1004.0         # J/kg/K
EPS    = RD / RV

C_SIGMA = 0.5            # seuil retenu dans le travail précédent (peak NSE tendance)

files = sorted(DATA_DIR.glob(FILE_PATTERN))
print(f"{len(files)} fichiers trouvés dans {DATA_DIR}")


In [ ]:

def load_level(files, varname, k, decode_times=False):
    '''Charge un niveau vertical k d'une variable pour tous les temps,
    fichier par fichier, pour éviter la saturation RAM sur le domaine complet.

    Retourne un tableau (nt, ny, nx).
    '''
    out = []
    for f in files:
        with nc.Dataset(f) as ds:
            var = ds.variables[varname]
            # ordre attendu des dims Méso-NH : (time, z, y, x)
            if var.ndim == 4:
                out.append(np.asarray(var[:, k, :, :]))
            elif var.ndim == 3:  # déjà un seul temps par fichier
                out.append(np.asarray(var[k, :, :])[None, ...])
            else:
                raise ValueError(f"Dimension inattendue pour {varname}: {var.shape}")
    return np.concatenate(out, axis=0)


def load_column(files, varname):
    '''Charge une variable indépendante de x,y,t (ex : niveaux verticaux ZHAT).'''
    with nc.Dataset(files[0]) as ds:
        return np.asarray(ds.variables[varname][:])


def rho0_profile(files, k_levels):
    '''Calcule rho0(z) via la température virtuelle et la loi des gaz parfaits,
    moyenné en x,y,t (profil de référence RCE).
    '''
    rho0 = np.zeros(len(k_levels))
    for i, k in enumerate(k_levels):
        th  = load_level(files, "THT", k)     # température potentielle
        rv  = load_level(files, "RVT", k)     # rapport de mélange vapeur
        p   = load_level(files, "PABST", k)   # pression
        thv = th * (1.0 + (RV / RD - 1.0) * rv / (1.0 + rv))
        T   = thv * (p / 1.0e5) ** (RD / CP)
        rho = p / (RD * T)
        rho0[i] = np.nanmean(rho)
    return rho0


In [ ]:

def compute_prw(files):
    '''Eau précipitable colonne (PRW) -> masques régions humides/sèches (mh/ms).'''
    zhat = load_column(files, "ZHAT")
    nz = len(zhat)
    prw = None
    for k in range(nz - 1):
        rv = load_level(files, "RVT", k)
        p  = load_level(files, "PABST", k)
        th = load_level(files, "THT", k)
        T  = th * (p / 1.0e5) ** (RD / CP)
        rho = p / (RD * T)
        dz = zhat[k + 1] - zhat[k]
        contrib = rho * rv * dz
        prw = contrib if prw is None else prw + contrib
    prw_mean_t = prw.mean(axis=0)
    thresh = prw_mean_t.mean()
    mh = prw_mean_t > thresh   # masque humide
    ms = ~mh                   # masque sec
    return prw, mh, ms


def three_category_mask(w, sigma_w, C=C_SIGMA):
    '''Catégorise chaque point en up / dn / env selon un seuil sur w
    (C * sigma_w), comme dans le pipeline validé (meilleur NSE que Romps 2014).
    '''
    up  = w >  C * sigma_w
    dn  = w < -C * sigma_w
    env = ~(up | dn)
    return up, dn, env


**Note** : les chemins/patterns ci-dessus sont des placeholders — à remplacer par les vrais chemins `large300`. Le reste du notebook suppose que `files`, `rho0_profile`, `load_level`, `three_category_mask` fonctionnent comme dans les notebooks précédents.

## 1. Visualisation — champs à un temps donné + évolution temporelle

But : un tableau de bord qui affiche, pour un temps $t$ donné (ou une moyenne glissante), les champs suivants sur une coupe $(x,z)$ (ou $(x,y)$ à un niveau donné) :

- flux de Reynolds $\rho_0\langle u'w'\rangle$
- cisaillement $\partial_z \bar u$
- vent $u$, vent $w$
- $\partial_z u$ (dérivée première)
- $\partial_z^2 u$ (dérivée seconde)

et en dessous, l'évolution temporelle (Hovmöller $z$–$t$, ou série temporelle à un niveau) de ces mêmes quantités.


In [ ]:

def vertical_derivative(field, zhat, axis=0):
    '''Dérivée verticale centrée (ordre 2), axis = indice de l'axe z dans field.'''
    return np.gradient(field, zhat, axis=axis, edge_order=2)


def reynolds_flux_profile(files, zhat, rho0):
    '''rho0(z) * <u'w'>(z,t), perturbations par rapport à la moyenne horizontale
    à chaque niveau et chaque instant.
    '''
    nz = len(zhat)
    nt = None
    flux = None
    ubar_prof, wbar_prof = [], []
    for k in range(nz):
        u = load_level(files, "UT", k)   # (nt, ny, nx)
        w = load_level(files, "WT", k)
        if nt is None:
            nt = u.shape[0]
            flux = np.zeros((nz, nt))
        ubar = u.mean(axis=(1, 2), keepdims=True)
        wbar = w.mean(axis=(1, 2), keepdims=True)
        up = u - ubar
        wp = w - wbar
        flux[k, :] = rho0[k] * np.mean(up * wp, axis=(1, 2))
        ubar_prof.append(ubar.squeeze())
        wbar_prof.append(wbar.squeeze())
    return flux, np.array(ubar_prof), np.array(wbar_prof)  # (nz,t)


In [ ]:

def dashboard_at_time(t_idx, zhat, flux, ubar, wbar):
    '''Panneau de diagnostics à un instant t_idx donné : profils verticaux.'''
    dudz  = vertical_derivative(ubar, zhat, axis=0)
    d2udz2 = vertical_derivative(dudz, zhat, axis=0)

    fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=True)
    panels = [
        (flux[:, t_idx],   r"$\rho_0\langle u'w'\rangle$ (kg m$^{-1}$s$^{-2}$)"),
        (dudz[:, t_idx],   r"$\partial_z \bar u$ (s$^{-1}$)"),
        (ubar[:, t_idx],   r"$\bar u$ (m/s)"),
        (wbar[:, t_idx],   r"$\bar w$ (m/s)"),
        (d2udz2[:, t_idx], r"$\partial_z^2 \bar u$ (m$^{-1}$s$^{-1}$)"),
    ]
    for ax, (field, label) in zip(axes, panels):
        ax.plot(field, zhat, lw=1.5)
        ax.axvline(0, color="k", lw=0.5)
        ax.set_xlabel(label)
    axes[0].set_ylabel("z (m)")
    fig.suptitle(f"Diagnostics — t_idx = {t_idx}")
    fig.tight_layout()
    return fig


def hovmoller_evolution(zhat, field, title, cmap="RdBu_r"):
    '''Hovmöller z-t pour une quantité de forme (nz, nt).'''
    vmax = np.nanpercentile(np.abs(field), 98)
    fig, ax = plt.subplots(figsize=(10, 4))
    im = ax.pcolormesh(np.arange(field.shape[1]), zhat, field,
                        cmap=cmap, vmin=-vmax, vmax=vmax, shading="auto")
    ax.set_xlabel("indice temporel")
    ax.set_ylabel("z (m)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    return fig

# Exemple d'utilisation (une fois flux, ubar, wbar, zhat calculés) :
# zhat = load_column(files, "ZHAT")
# rho0 = rho0_profile(files, range(len(zhat)))
# flux, ubar, wbar = reynolds_flux_profile(files, zhat, rho0)
# dashboard_at_time(t_idx=10, zhat=zhat, flux=flux, ubar=ubar, wbar=wbar)
# hovmoller_evolution(zhat, flux, r"$\rho_0\langle u'w'\rangle$ (z,t)")


## 2. Equation discovery — termes physiques grande échelle uniquement

Contrainte : la bibliothèque de candidats ne doit contenir **que des termes calculables à partir des champs résolus à grande échelle** (moyennes / profils), pas de termes de sous-maille (variance, flux turbulent local, etc.) qui seraient eux-mêmes ce qu'on cherche à paramétrer.

Bibliothèque proposée : $\{\bar u,\ \partial_z \bar u,\ \partial_z^2 \bar u,\ \bar w,\ \bar u \partial_z \bar u,\ z\,\partial_z\bar u,\ \bar u^2\}$ — à ajuster selon ce qui a du sens physiquement.

On implémente STLSQ (Sequential Thresholded Least-Squares) "à la main" pour garder le contrôle total sur le seuillage, dans la continuité du travail déjà fait (au lieu de dépendre de `pysindy`).


In [ ]:

def build_candidate_library(ubar, wbar, zhat):
    '''Construit la bibliothèque de termes grande échelle, forme (nz*nt, n_terms).
    ubar, wbar : (nz, nt). zhat : (nz,).
    '''
    nz, nt = ubar.shape
    dudz   = vertical_derivative(ubar, zhat, axis=0)
    d2udz2 = vertical_derivative(dudz, zhat, axis=0)
    Z = np.tile(zhat[:, None], (1, nt))

    terms = {
        "u":         ubar,
        "dudz":      dudz,
        "d2udz2":    d2udz2,
        "w":         wbar,
        "u_dudz":    ubar * dudz,
        "z_dudz":    Z * dudz,
        "u2":        ubar ** 2,
    }
    names = list(terms.keys())
    Theta = np.stack([terms[n].ravel() for n in names], axis=1)  # (nz*nt, n_terms)
    return Theta, names


def stlsq(Theta, y, threshold=0.05, n_iter=15, alpha=1e-8):
    '''Sequential Thresholded Least-Squares (SINDy-style).
    Theta : (N, n_terms) déjà standardisée si besoin. y : (N,).
    Retourne le vecteur de coefficients (n_terms,) et le masque des termes actifs.
    '''
    n_terms = Theta.shape[1]
    # régression ridge légère pour la stabilité numérique
    XtX = Theta.T @ Theta + alpha * np.eye(n_terms)
    Xty = Theta.T @ y
    coef = np.linalg.solve(XtX, Xty)
    active = np.ones(n_terms, dtype=bool)

    for _ in range(n_iter):
        small = np.abs(coef) < threshold
        if not np.any(small & active):
            break
        active[small] = False
        if not np.any(active):
            break
        Xs = Theta[:, active]
        XtXs = Xs.T @ Xs + alpha * np.eye(Xs.shape[1])
        Xtys = Xs.T @ y
        coef_active = np.linalg.solve(XtXs, Xtys)
        coef = np.zeros(n_terms)
        coef[active] = coef_active

    return coef, active


def normalize_library(Theta):
    '''Standardise chaque colonne (moyenne 0, écart-type 1) et renvoie les facteurs
    pour pouvoir dénormaliser les coefficients ensuite.
    '''
    mu = Theta.mean(axis=0)
    sigma = Theta.std(axis=0) + 1e-12
    return (Theta - mu) / sigma, mu, sigma


In [ ]:

def run_equation_discovery(flux, ubar, wbar, zhat, threshold=0.05):
    '''Applique STLSQ pour expliquer rho0<u'w'> par la bibliothèque grande échelle.'''
    Theta, names = build_candidate_library(ubar, wbar, zhat)
    y = flux.ravel()

    Theta_n, mu, sigma = normalize_library(Theta)
    y_n = (y - y.mean()) / (y.std() + 1e-12)

    coef_n, active = stlsq(Theta_n, y_n, threshold=threshold)

    # dénormalisation approximative pour lecture physique
    coef = coef_n * (y.std() / sigma)

    print("Termes retenus :")
    for name, c, a in zip(names, coef, active):
        if a:
            print(f"  {name:12s} : {c:+.4e}")

    y_pred_n = Theta_n @ coef_n
    r2 = r2_score(y_n, y_pred_n)
    print(f"\nR^2 (normalisé) = {r2:.3f}")
    return dict(names=names, coef=coef, active=active, r2=r2, Theta=Theta)


# Exemple :
# result = run_equation_discovery(flux, ubar, wbar, zhat, threshold=0.08)


## 3. Robustesse du split train/test dans le temps

**Problème identifié** : faire des splits train/test sur des pas de temps différents pour deviner des grandeurs (flux de Reynolds) qui *dérivent au cours du temps* est optimiste — un split aléatoire mélange des instants voisins fortement corrélés entre train et test, et surestime la performance ; en RCE le signal peut aussi présenter une dérive lente (spin-up, oscillations basse fréquence).

Pistes :
- **CV temporelle stricte** (walk-forward / blocked) : `sklearn.model_selection.TimeSeriesSplit`, jamais de fuite du futur vers le passé.
- **Test de stationnarité** du flux de Reynolds moyen (ou par niveau) pour quantifier la dérive.
- **Autres techniques ML** à tester si STLSQ/régression linéaire sature : gradient boosting (capture les non-linéarités sans sur-ajuster autant qu'un réseau profond sur peu de données), symbolic regression (Gplearn/PySR) pour rester interprétable, ou modèles à état (ex. filtre de Kalman / SSM) si la dérive temporelle elle-même a une dynamique exploitable.


In [ ]:

def check_stationarity(y_t, window=50):
    '''Diagnostic simple de dérive : moyenne glissante + test de rupture naïf
    (comparaison de la moyenne sur la 1ere vs derniere fenetre).'''
    roll_mean = np.convolve(y_t, np.ones(window) / window, mode="valid")
    first, last = y_t[:window].mean(), y_t[-window:].mean()
    drift = (last - first) / (np.abs(first) + 1e-12)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(y_t, alpha=0.4, label="signal brut")
    ax.plot(np.arange(window // 2, window // 2 + len(roll_mean)), roll_mean,
            lw=2, label=f"moyenne glissante ({window})")
    ax.legend()
    ax.set_title(f"Dérive relative début→fin : {drift:+.1%}")
    fig.tight_layout()
    return drift, fig


In [ ]:

def temporal_cv_benchmark(Theta, y, n_splits=5, models=None):
    '''Compare un split temporel strict (walk-forward) à un split aléatoire,
    pour un ou plusieurs modèles.
    '''
    if models is None:
        models = {
            "Linear": LinearRegression(),
            "GradBoost": GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                                     learning_rate=0.05),
        }

    tscv = TimeSeriesSplit(n_splits=n_splits)
    results = {name: {"temporal": [], "random": []} for name in models}

    rng = np.random.default_rng(0)
    n = len(y)
    idx_shuffled = rng.permutation(n)

    for name, model in models.items():
        # --- split temporel strict ---
        for train_idx, test_idx in tscv.split(Theta):
            model.fit(Theta[train_idx], y[train_idx])
            pred = model.predict(Theta[test_idx])
            results[name]["temporal"].append(r2_score(y[test_idx], pred))

        # --- split aléatoire (référence "optimiste") ---
        fold_size = n // n_splits
        for i in range(n_splits):
            test_idx = idx_shuffled[i * fold_size:(i + 1) * fold_size]
            train_idx = np.setdiff1d(idx_shuffled, test_idx)
            model.fit(Theta[train_idx], y[train_idx])
            pred = model.predict(Theta[test_idx])
            results[name]["random"].append(r2_score(y[test_idx], pred))

    for name in models:
        t_mean = np.mean(results[name]["temporal"])
        r_mean = np.mean(results[name]["random"])
        print(f"{name:10s} | R^2 walk-forward = {t_mean:+.3f}  |  "
              f"R^2 random-split = {r_mean:+.3f}  (écart = {r_mean - t_mean:+.3f})")
    return results

# Exemple :
# Theta_flat = Theta  # (nz*nt, n_terms) déjà construit à l'objectif 2
# y_flat = flux.ravel()
# temporal_cv_benchmark(Theta_flat, y_flat)


## 4. Flux moyenné en y → fonction de courant → POD/ACP

$$\bar\Phi(x,z,t) = \langle \rho_0 u'w' \rangle_y$$

On calcule la fonction de courant $\psi$ associée à l'écoulement moyen-y perturbé (solveur de Poisson, périodique en $x$, Dirichlet en $z$ — cohérent avec les notebooks précédents), puis on fait une décomposition POD/ACP :

$$\psi(x,z,t) \approx \bar\psi(x,z) + \sum_n a_n(t)\,\psi_n(x,z)$$

avec reconstruction de $u,w$ via $\tilde u_N = \partial_z \tilde\psi_N$, $\tilde w_N = -\partial_x \tilde\psi_N$, et vérification que $\overline{\tilde w'_N \tilde u'_N} \approx \overline{w'u'}$.

Le nombre de modes à garder ("tête de l'ACP") est choisi par critère d'énergie cumulée (ex. 90-95%) **et** vérifié par erreur de reconstruction hors-échantillon (CV temporelle, cf. objectif 3, pour éviter d'overfitter le nombre de modes sur les mêmes instants que ceux utilisés pour l'évaluation).


In [ ]:

def poisson_streamfunction(u, w, x, z):
    '''Résout nabla^2 psi = -omega_y, avec omega_y = du/dz - dw/dx,
    conditions périodiques en x, Dirichlet (psi=0) en z (haut et bas).
    u, w : (nz, nx) à un instant donné. x, z : coordonnées 1D.
    '''
    nz, nx = u.shape
    dz = np.gradient(z)
    dx = x[1] - x[0]

    dudz = np.gradient(u, z, axis=0, edge_order=2)
    dwdx = np.gradient(w, x, axis=1, edge_order=2)
    omega = dudz - dwdx  # vorticité y

    # Construction de l'opérateur laplacien discret (différences finies,
    # périodique en x via indices modulo, Dirichlet en z via psi=0 aux bords).
    N = nz * nx
    A = sparse.lil_matrix((N, N))
    b = np.zeros(N)

    def idx(k, i):
        return k * nx + i

    dz_mean = np.mean(dz)
    for k in range(nz):
        for i in range(nx):
            row = idx(k, i)
            if k == 0 or k == nz - 1:
                A[row, row] = 1.0
                b[row] = 0.0
                continue
            ip = (i + 1) % nx
            im = (i - 1) % nx
            A[row, idx(k, i)]  = -2.0 / dx**2 - 2.0 / dz_mean**2
            A[row, idx(k, ip)] = 1.0 / dx**2
            A[row, idx(k, im)] = 1.0 / dx**2
            A[row, idx(k + 1, i)] = 1.0 / dz_mean**2
            A[row, idx(k - 1, i)] = 1.0 / dz_mean**2
            b[row] = -omega[k, i]

    A = A.tocsc()
    lu = splu(A)
    psi = lu.solve(b).reshape(nz, nx)
    return psi, omega


In [ ]:

def pod_decomposition(psi_series, x, z, energy_target=0.95):
    '''POD/ACP de psi'(x,z,t). psi_series : (nt, nz, nx) déjà anomalies
    (moyenne temporelle retirée).
    Retourne : modes psi_n (n_modes, nz, nx), coefficients a_n(t) (nt, n_modes),
    variance expliquée cumulée, et n_modes retenu pour atteindre energy_target.
    '''
    nt, nz, nx = psi_series.shape
    X = psi_series.reshape(nt, nz * nx)

    pca = PCA(n_components=min(nt, nz * nx))
    a_n = pca.fit_transform(X)          # (nt, n_components)
    modes = pca.components_.reshape(-1, nz, nx)  # (n_components, nz, nx)
    cum_energy = np.cumsum(pca.explained_variance_ratio_)

    n_modes = int(np.searchsorted(cum_energy, energy_target) + 1)
    print(f"{n_modes} modes nécessaires pour {energy_target:.0%} d'énergie "
          f"(sur {len(cum_energy)} modes possibles).")
    return modes, a_n, cum_energy, n_modes, pca


def plot_pod_energy(cum_energy, n_modes):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(np.arange(1, len(cum_energy) + 1), cum_energy, "o-", ms=3)
    ax.axhline(cum_energy[n_modes - 1], color="grey", ls="--", lw=0.8)
    ax.axvline(n_modes, color="grey", ls="--", lw=0.8)
    ax.set_xlabel("nombre de modes")
    ax.set_ylabel("énergie cumulée expliquée")
    ax.set_title("Critère de troncature POD")
    fig.tight_layout()
    return fig


In [ ]:

def reconstruct_uw_from_modes(modes, a_n, x, z, n1, n2):
    '''Reconstruit u~_N, w~_N a partir des modes n1..n2 (bornes incluses,
    indexation a partir de 0) : u~ = d(psi)/dz, w~ = -d(psi)/dx.
    '''
    n_modes_used = modes[n1:n2 + 1]                 # (n_sel, nz, nx)
    a_sel = a_n[:, n1:n2 + 1]                        # (nt, n_sel)
    psi_N = np.einsum("tn,nzx->tzx", a_sel, n_modes_used)  # (nt, nz, nx)

    u_N = np.gradient(psi_N, z, axis=1, edge_order=2)
    w_N = -np.gradient(psi_N, x, axis=2, edge_order=2)
    return psi_N, u_N, w_N


def check_reynolds_flux_reconstruction(u_N, w_N, u_true, w_true):
    '''Compare <w'_N u'_N> (à partir des N modes retenus) a <w'u'> complet,
    par niveau z, moyenne sur x et t.
    '''
    up_N = u_N - u_N.mean(axis=(0, 2), keepdims=True)
    wp_N = w_N - w_N.mean(axis=(0, 2), keepdims=True)
    flux_N = np.mean(up_N * wp_N, axis=(0, 2))

    up_t = u_true - u_true.mean(axis=(0, 2), keepdims=True)
    wp_t = w_true - w_true.mean(axis=(0, 2), keepdims=True)
    flux_true = np.mean(up_t * wp_t, axis=(0, 2))

    nse = 1 - np.sum((flux_N - flux_true) ** 2) / np.sum((flux_true - flux_true.mean()) ** 2)
    print(f"NSE reconstruction flux (modes utilisés) = {nse:.3f}")
    return flux_N, flux_true, nse


In [ ]:

def mode_energy_content(a_n, n1, n2, dt_windows=None):
    '''psi1^2 = (1/Delta t) * sum_t sum_n a_n(t)^2 * ||psi_n||^2, formule notée
    dans les notes, pour caractériser le contenu energetique des modes n1..n2
    par bloc temporel (dt_windows : liste de slices, sinon tout l'intervalle).
    '''
    norms2 = np.array([np.sum(a_n[:, n] ** 2) for n in range(n1, n2 + 1)])  # proxy ||psi_n||^2 via a_n si modes normalises
    if dt_windows is None:
        dt_windows = [slice(0, a_n.shape[0])]

    energies = []
    for sl in dt_windows:
        a_block = a_n[sl, n1:n2 + 1]
        e = np.sum(a_block ** 2 * norms2[None, :]) / a_block.shape[0]
        energies.append(e)
    return np.array(energies)

# Exemple d'enchainement objectif 4 :
# psi_t = np.stack([poisson_streamfunction(u[k], w[k], x, z)[0] for k in range(nt)])
# psi_anom = psi_t - psi_t.mean(axis=0, keepdims=True)
# modes, a_n, cum_energy, n_modes, pca = pod_decomposition(psi_anom, x, z)
# plot_pod_energy(cum_energy, n_modes)
# psi_N, u_N, w_N = reconstruct_uw_from_modes(modes, a_n, x, z, 0, n_modes - 1)
# check_reynolds_flux_reconstruction(u_N, w_N, u_true, w_true)


## 5. Circulation — cellules fermées de $\psi$

Idée des notes : une **circulation fermée** part de $\psi=0$ et y revient avec un **signe unique** de $\psi$ tout le long du contour. On peut exploiter ce fait (signe constant) pour segmenter le domaine en cellules de circulation de façon exacte, plutôt que par détection d'extrema + ellipse fitting (méthode précédente, moins directe).

Approche : à un instant donné, on prend $\text{sign}(\psi')$, on labellise les composantes connexes de chaque signe (`skimage.measure.label`), et chaque composante connexe correspond à une cellule de circulation candidate. On calcule ensuite la circulation $\Gamma$ de chaque cellule par intégrale de la vorticité sur la cellule (théorème de Stokes, $\Gamma = \iint \omega_y \, dx\,dz$), ce qui est équivalent à l'intégrale de contour $\oint \mathbf v \cdot d\mathbf l$ mais numériquement plus stable.


In [ ]:

def circulation_cells(psi, omega, min_size=9):
    '''Segmente psi (nz, nx) en cellules de circulation par signe constant,
    filtre les cellules trop petites (bruit), calcule Gamma = somme(omega * dA)
    par cellule.
    '''
    sign_map = np.sign(psi)
    sign_map[sign_map == 0] = 1  # évite un 3e label pour psi=0 exact

    cells = []
    for s in (1, -1):
        mask = sign_map == s
        lbl = label(mask, connectivity=2)
        for region in regionprops(lbl):
            if region.area < min_size:
                continue
            coords = region.coords
            gamma = omega[coords[:, 0], coords[:, 1]].sum()
            cells.append(dict(sign=s, area=region.area, gamma=gamma,
                               centroid=region.centroid, coords=coords))
    cells.sort(key=lambda c: -abs(c["gamma"]))
    return cells, sign_map


def plot_circulation_cells(psi, cells, x, z, top_n=10):
    fig, ax = plt.subplots(figsize=(9, 5))
    vmax = np.nanpercentile(np.abs(psi), 98)
    im = ax.pcolormesh(x, z, psi, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto")
    for c in cells[:top_n]:
        cz, cx = c["centroid"]
        ax.plot(x[int(cx)], z[int(cz)], "o", color="k", ms=4)
        ax.annotate(f"{c['gamma']:.1e}", (x[int(cx)], z[int(cz)]),
                    fontsize=7, color="k")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("z (m)")
    ax.set_title(f"Cellules de circulation (top {top_n} par |Gamma|)")
    fig.colorbar(im, ax=ax, label=r"$\psi'$")
    fig.tight_layout()
    return fig


def circulation_vs_time(psi_series, omega_series, min_size=9):
    '''Applique circulation_cells a chaque instant, retourne le nombre de
    cellules significatives et la circulation totale (|Gamma| sommee) au cours du temps.
    '''
    n_cells_t, gamma_tot_t = [], []
    for t in range(psi_series.shape[0]):
        cells, _ = circulation_cells(psi_series[t], omega_series[t], min_size=min_size)
        n_cells_t.append(len(cells))
        gamma_tot_t.append(sum(abs(c["gamma"]) for c in cells))
    return np.array(n_cells_t), np.array(gamma_tot_t)

# Exemple :
# psi0, omega0 = poisson_streamfunction(u[0], w[0], x, z)
# cells, sign_map = circulation_cells(psi0 - psi0.mean(), omega0)
# plot_circulation_cells(psi0 - psi0.mean(), cells, x, z)


## Synthèse / prochaines étapes

- **Obj. 1** : dashboard prêt, à brancher sur les vrais chemins de fichiers.
- **Obj. 2** : STLSQ opérationnel sur bibliothèque grande échelle ; à itérer sur le choix du `threshold` et à croiser avec des sous-domaines (up/dn/env) pour voir si les termes retenus diffèrent par catégorie.
- **Obj. 3** : `temporal_cv_benchmark` quantifie l'écart optimiste du split aléatoire vs walk-forward — point de vigilance à documenter dans le rapport ; Gradient Boosting comme premier candidat non-linéaire interprétable-ish, symbolic regression à évaluer si besoin de rester lisible physiquement.
- **Obj. 4** : critère d'énergie cumulée codé, mais il faut coupler ce choix de `n_modes` à une validation par NSE de reconstruction du flux de Reynolds (`check_reynolds_flux_reconstruction`), idéalement en walk-forward (obj. 3) pour éviter de choisir le nombre de modes sur les mêmes données que celles servant à l'évaluer.
- **Obj. 5** : la segmentation par signe constant est plus directe que l'ancienne méthode par extrema + ellipse, mais reste sensible à `min_size` (bruit numérique du solveur de Poisson) — à valider par comparaison qualitative avec les figures d'extrema déjà produites.
